In [1]:
from PIL import Image #open imgs for v1, v2
import matplotlib.pyplot as plt #plot imgs
import cv2 #for v3
import os #file and image management
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import re #regular expressions
import torch
import numpy as np
import sys

In [2]:
import transformers

In [3]:
from transformers import pipeline #easy pipeline for DA models



In [4]:
device = "cuda" if torch.cuda.is_available() else "cpu" #set device as GPU if available

models = {
    "V1": "LiheYoung/depth-anything-base-hf",
    "V2": "depth-anything/Depth-Anything-V2-base-hf",
    # "V3": "qualcomm/Depth-Anything-V3"
}


    
pipelines = {}
print("\nLoading models")
for version, repo in models.items():
    try:
        pipelines[version] = pipeline(task="depth-estimation", model=repo, device=device, torch_dtype=torch.float32)
        print(f" {version} loaded successfully.")
    except Exception as e:
        print(f" Couldn't Load {version}.")


# v3_model_path = os.path.join(v3_root, "checkpoints", "DA3NESTED-BASE") 




Loading models


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

The image processor of type `DPTImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


 V1 loaded successfully.


Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

 V2 loaded successfully.


In [5]:

current_dir = os.getcwd()
v3_folder = os.path.join(current_dir, "depth-anything-3")
v3_src = os.path.join(v3_folder, "src")

if v3_src not in sys.path:
    sys.path.insert(0, v3_src)

# set device
device = "cuda" if torch.cuda.is_available() else "cpu"

print("--- loading depth anything v3 local (base) ---")
try:
    # now that the folder is next to the notebook, this should work
    from depth_anything_3.api import DepthAnything3
    
    # attempt to load the base model
    # note: if you get a 401 error, we will need to point to a local .pth file
    model = DepthAnything3.from_pretrained("depth-anything/DA3-BASE")
    model = model.to(device=torch.device(device))
    
    pipelines["V3"] = model
    print(" v3 base loaded successfully from local folder.")
except ModuleNotFoundError as e:
    print(f" import error: {e}. check if 'depth-anything-3/src/depth_anything_3' exists.")
except Exception as e:
    print(f" load failed: {e}")

--- loading depth anything v3 local (base) ---
[WARN ] Dependency `gsplat` is required for rendering 3DGS. Install via: pip install git+https://github.com/nerfstudio-project/gsplat.git@0b4dddf04cb687367602c01196913cde6a743d70


WARNING[XFORMERS]: xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.7.1+cu126 with CUDA 1208 (you have 2.7.1+cu118)
    Python  3.9.13 (you have 3.10.10)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


[INFO ] using MLP layer as FFN
 v3 base loaded successfully from local folder.


In [11]:
def compare_single_image(image_path, save_path=None):
    """
    runs all available models (v1, v2, v3) and plots them with unified color logic
    """
    try:
        available_models = sorted(pipelines.keys())
        if not available_models: return

        # setup naming
        base_name = os.path.splitext(os.path.basename(image_path))[0]
        clean_name = re.sub(r'\d+', '', base_name).replace('_', ' ').strip().title()

        img_pil = Image.open(image_path).convert("RGB")
        img_np = np.array(img_pil)
        
        results = {}
        for version in available_models:
            model = pipelines[version]
            if version == "V3":
                prediction = model.inference([img_np])
                # v3 output is often disparity (higher = closer)
                # we keep it raw here and handle logic in the plotting loop
                results[version] = prediction.depth[0]
            else:
                res = model(img_pil, timeout=None)
                results[version] = res["predicted_depth"].squeeze().cpu().numpy()

        num_plots = len(results) + 1
        fig, axes = plt.subplots(1, num_plots, figsize=(5 * num_plots, 5))
        if num_plots == 1: axes = [axes]
        fig.suptitle(clean_name, fontsize=14, fontweight='bold', y=0.98)

        # 0. original image
        axes[0].imshow(img_pil)
        axes[0].set_title("Original Image", fontsize=10, fontweight='bold')
        axes[0].axis('off')

        # 1-N. depth maps
        for i, version in enumerate(available_models):
            depth_map_ptr = np.array(results[version])
            
            # basic normalization: 0 to 1
            d_min, d_max = depth_map_ptr.min(), depth_map_ptr.max()
            depth_norm = (depth_map_ptr - d_min) / (d_max - d_min + 1e-8)

            # logical check for v3
            # if the floor/bottom of the ames room is darker than the ceiling, it's inverted
            # standard: closer objects (bottom of image usually) should be lighter (yellow/white)
            if version == "V3":
                # we force v3 to match v1/v2 orientation
                # usually v1/v2: close = high value (light)
                # if your v3 is showing close = dark, we flip it
                # current evidence shows v3 needs flipping to match v1/v2 behavior
                depth_norm = 1.0 - depth_norm

            axes[i+1].imshow(depth_norm, cmap='inferno', interpolation="bilinear")
            axes[i+1].set_title(f"Depth Anything {version}", fontsize=10, fontweight='bold')
            axes[i+1].axis('off')

        plt.tight_layout(rect=[0, 0, 1, 0.92])

        if save_path:
            plt.savefig(save_path, dpi=200, bbox_inches='tight')
            plt.close(fig)
        else:
            plt.show()

    except Exception as e:
        print(f" error in compare_single_image: {e}")

def process_dataset(input_folder="3D_illusions", output_folder="results"):
    """
    automates processing for the entire directory structure
    """
    # final check before starting
    print(f"models currently loaded: {list(pipelines.keys())}")
    # print(f"starting batch processing...")
    
    total = 0
    if not os.path.exists(input_folder):
        print(f" error: input folder '{input_folder}' not found.")
        return

    for root, dirs, files in os.walk(input_folder):
        for filename in files:
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(root, filename)
                
                # maintain subdirectory structure
                rel_path = os.path.relpath(root, input_folder)
                target_dir = os.path.join(output_folder, rel_path)
                os.makedirs(target_dir, exist_ok=True)
                
                save_path = os.path.join(target_dir, f"cmp_{filename}")
                
                print(f" processing -> {filename}")
                compare_single_image(img_path, save_path=save_path)
                total += 1

    print(f"dataset processing complete. total images: {total}")

# execute
process_dataset()

models currently loaded: ['V1', 'V2', 'V3']
 processing -> corridor_illusion.jpg
[INFO ] Processed Images Done taking 0.009972810745239258 seconds. Shape:  torch.Size([1, 3, 504, 504])
[INFO ] Model Forward Pass Done. Time: 0.07380247116088867 seconds
[INFO ] Conversion to Prediction Done. Time: 0.0009629726409912109 seconds
 processing -> figure_ground.png
[INFO ] Processed Images Done taking 0.008975744247436523 seconds. Shape:  torch.Size([1, 3, 504, 476])
[INFO ] Model Forward Pass Done. Time: 0.07700181007385254 seconds
[INFO ] Conversion to Prediction Done. Time: 0.000997304916381836 seconds
 processing -> kanizsa_triangle.png
[INFO ] Processed Images Done taking 0.008978605270385742 seconds. Shape:  torch.Size([1, 3, 476, 504])
[INFO ] Model Forward Pass Done. Time: 0.0712435245513916 seconds
[INFO ] Conversion to Prediction Done. Time: 0.022938966751098633 seconds
 processing -> Müller-Lyer_illusion.png
[INFO ] Processed Images Done taking 0.005983829498291016 seconds. Shape:  

In [10]:
process_dataset(input_folder="synthetic data/corridor_illusion", output_folder="synthetic_results")

models currently loaded: ['V1', 'V2', 'V3']
 processing -> base_image.png
[INFO ] Processed Images Done taking 0.028921842575073242 seconds. Shape:  torch.Size([1, 3, 504, 504])
[INFO ] Model Forward Pass Done. Time: 0.06757259368896484 seconds
[INFO ] Conversion to Prediction Done. Time: 0.000997781753540039 seconds
 processing -> cube_walls.jpg
[INFO ] Processed Images Done taking 0.013963460922241211 seconds. Shape:  torch.Size([1, 3, 504, 504])
[INFO ] Model Forward Pass Done. Time: 0.08509469032287598 seconds
[INFO ] Conversion to Prediction Done. Time: 0.000997304916381836 seconds
 processing -> cylinder_pattern.png
[INFO ] Processed Images Done taking 0.026926517486572266 seconds. Shape:  torch.Size([1, 3, 504, 504])
[INFO ] Model Forward Pass Done. Time: 0.07308721542358398 seconds
[INFO ] Conversion to Prediction Done. Time: 0.000997781753540039 seconds
 processing -> cylinder_pattern_bordered.png
[INFO ] Processed Images Done taking 0.012859106063842773 seconds. Shape:  torch